In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from torchvision import transforms
from tqdm import tqdm

c:\Users\siddh\Downloads\SBI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IMAGE_DIR = r"C:\Users\siddh\Downloads\SBI\sop project\images"
CSV_PATH = r"C:\Users\siddh\Downloads\SBI\sop project\df_trainset_midi.csv"
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4

In [3]:
df = pd.read_csv(CSV_PATH)

# Drop rows with missing pore%
df = df.dropna(subset=["pore%"]).reset_index(drop=True)

print("Total samples:", len(df))
print(df.head())

Total samples: 1135
   Unnamed: 0 Image_Name  Area_avg  Area_max  Area_std  Aspect_ratio_avg  \
0          10      4_4_1    82.961   310.859    72.815             1.194   
1           1      4_1_1    66.993   449.018    79.834             1.192   
2           2      4_1_2    44.134    85.198    30.415             1.131   
3           3      4_2_1    55.264   283.227    70.037             1.264   
4           4      4_2_2   104.175   693.100   154.974             1.336   

   Roundness_avg  Roundness_std  Max angle  Uniqueness_avg  ...  \
0          0.869          0.061       96.0           0.734  ...   
1          0.856          0.053      101.0           0.680  ...   
2          0.850          0.063       90.0           0.687  ...   
3          0.826          0.076      104.0           0.619  ...   
4          0.823          0.113      104.0           0.746  ...   

   Normalized_Enthalpy          mu     sigma  skewness  kurtosis  \
0             8.379002  816.483607  23.72693 -0.3713

In [4]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(df, groups=df["Sample"])
)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print("Train samples:", train_df["Sample"].nunique())
print("Val samples:", val_df["Sample"].nunique())
print(
    "Overlap:",
    set(train_df["Sample"]).intersection(set(val_df["Sample"]))
)

Train samples: 58
Val samples: 15
Overlap: set()


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [6]:
class PorosityDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = str(row["Image_Name"]).strip()

        # ensure .tif extension
        if not img_name.lower().endswith(".tif"):
            img_name = img_name + ".tif"

        img_path = os.path.join(self.image_dir, img_name)

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Missing file: {img_path}")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row["pore%"], dtype=torch.float32)

        return image, target

In [7]:
train_dataset = PorosityDataset(train_df, IMAGE_DIR, train_transform)
val_dataset = PorosityDataset(val_df, IMAGE_DIR, train_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = timm.create_model("efficientnet_b2", pretrained=True)

# Replace classifier for regression
model.classifier = nn.Linear(model.classifier.in_features, 1)

model = model.to(device)

Using device: cuda


In [9]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
best_r2 = -float("inf")
for epoch in range(EPOCHS):
    model.train()
    train_losses = []

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        images = images.to(device)
        targets = targets.to(device).unsqueeze(1)

        outputs = model(images)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ===== validation =====
    model.eval()
    preds = []
    gts = []

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Val"):
            images = images.to(device)
            targets = targets.to(device).unsqueeze(1)

            outputs = model(images)

            preds.extend(outputs.cpu().numpy().flatten())
            gts.extend(targets.cpu().numpy().flatten())

    mae = mean_absolute_error(gts, preds)
    r2 = r2_score(gts, preds)
    if r2 > best_r2:
        best_r2 = r2
        torch.save(model.state_dict(), "best_efficientnet_b2_porosity.pth")
        print(f"✅ Saved new best model (R2={r2:.4f})")

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {np.mean(train_losses):.4f}")
    print(f"Val MAE: {mae:.4f}")
    print(f"Val R2: {r2:.4f}")

Val: 100%|██████████| 15/15 [00:01<00:00, 12.71it/s]


✅ Saved new best model (R2=0.6101)

Epoch 1
Train Loss: 0.0368
Val MAE: 0.0799
Val R2: 0.6101


Val: 100%|██████████| 15/15 [00:01<00:00, 12.52it/s]


✅ Saved new best model (R2=0.7210)

Epoch 2
Train Loss: 0.0110
Val MAE: 0.0645
Val R2: 0.7210


Val: 100%|██████████| 15/15 [00:01<00:00, 12.45it/s]


✅ Saved new best model (R2=0.8261)

Epoch 3
Train Loss: 0.0086
Val MAE: 0.0523
Val R2: 0.8261


... (20 outputs left out)

Val: 100%|██████████| 15/15 [00:01<00:00, 12.05it/s]



Epoch 14
Train Loss: 0.0017
Val MAE: 0.0277
Val R2: 0.9487


Val: 100%|██████████| 15/15 [00:01<00:00, 11.75it/s]

✅ Saved new best model (R2=0.9632)

Epoch 15
Train Loss: 0.0023
Val MAE: 0.0233
Val R2: 0.9632


In [10]:
torch.save(model.state_dict(), "efficientnet_b2_porosity.pth")
print("Model saved.")

Model saved.


In [11]:
print(train_df["Sample"].nunique())
print(val_df["Sample"].nunique())

print(
    set(train_df["Sample"]).intersection(
        set(val_df["Sample"])
    )
)

58
15
set()
